In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [2]:
ookla_df = pd.read_csv('Ookla_20.csv')

ookla_df

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry
0,22133222312322,5227.000000,2449.000000,54.000000,2,2,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,22133222313202,8630.000000,3638.000000,44.000000,1,1,"POLYGON ((-160.0048828125 70.6472305085909, -1..."
2,22133222330010,9529.000000,3214.000000,44.000000,1,1,"POLYGON ((-160.037841796875 70.6417687358462, ..."
3,22133222330011,9392.000000,3390.000000,42.000000,1,1,"POLYGON ((-160.032348632812 70.6417687358462, ..."
4,22133222330013,961.000000,723.000000,50.000000,2,1,"POLYGON ((-160.032348632812 70.6399478155463, ..."
...,...,...,...,...,...,...,...
7620210,3131120221103110,9752.222222,9283.444444,6.000000,1,1,"POLYGON ((169.354248046875 -46.5890690830918, ..."
7620211,3131120300000113,3081.888889,1432.000000,3.888889,1,1,"POLYGON ((170.194702148438 -46.0770413127077, ..."
7620212,3131120300000300,10736.250000,5517.500000,12.000000,14,0,"POLYGON ((170.17822265625 -46.0884717957759, 1..."
7620213,3301200301201233,10110.000000,5192.000000,11.000000,1,1,"POLYGON ((114.318237304688 -71.1434194049621, ..."


In [3]:
viirs_cleaned_agg_df = pd.read_csv('viirs_20_cleaned_agg.csv')

viirs_cleaned_agg_df

,longitude,latitude,light_intensity
0,-53.133333,-22.645834,0.023611
1,-53.133333,-22.641667,0.023750
2,-53.133333,-22.637500,0.023194
3,-53.129167,-22.650000,0.023542
4,-53.129167,-22.645834,0.024028
...,...,...,...
4219474,-37.354166,-11.595833,0.020347
4219475,-37.354166,-11.591667,0.020764
4219476,-37.354166,-11.587500,0.021042
4219477,-37.354166,-11.583333,0.021806


In [4]:
# Convert Ookla geometry to GeoDataFrame
ookla_df['geometry'] = ookla_df['geometry'].apply(wkt.loads)
ookla_gdf = gpd.GeoDataFrame(ookla_df, geometry='geometry')

ookla_gdf

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry
0,22133222312322,5227.000000,2449.000000,54.000000,2,2,"POLYGON ((-160.02686 70.64359, -160.02136 70.6..."
1,22133222313202,8630.000000,3638.000000,44.000000,1,1,"POLYGON ((-160.00488 70.64723, -159.99939 70.6..."
2,22133222330010,9529.000000,3214.000000,44.000000,1,1,"POLYGON ((-160.03784 70.64177, -160.03235 70.6..."
3,22133222330011,9392.000000,3390.000000,42.000000,1,1,"POLYGON ((-160.03235 70.64177, -160.02686 70.6..."
4,22133222330013,961.000000,723.000000,50.000000,2,1,"POLYGON ((-160.03235 70.63995, -160.02686 70.6..."
...,...,...,...,...,...,...,...
7620210,3131120221103110,9752.222222,9283.444444,6.000000,1,1,"POLYGON ((169.35425 -46.58907, 169.35974 -46.5..."
7620211,3131120300000113,3081.888889,1432.000000,3.888889,1,1,"POLYGON ((170.1947 -46.07704, 170.2002 -46.077..."
7620212,3131120300000300,10736.250000,5517.500000,12.000000,14,0,"POLYGON ((170.17822 -46.08847, 170.18372 -46.0..."
7620213,3301200301201233,10110.000000,5192.000000,11.000000,1,1,"POLYGON ((114.31824 -71.14342, 114.32373 -71.1..."


In [5]:
# Convert VIIRS longitude/latitude to GeoDataFrame
viirs_cleaned_agg_gdf = gpd.GeoDataFrame(
    viirs_cleaned_agg_df, geometry=gpd.points_from_xy(viirs_cleaned_agg_df.longitude, viirs_cleaned_agg_df.latitude)
)

viirs_cleaned_agg_gdf

,longitude,latitude,light_intensity,geometry
0,-53.133333,-22.645834,0.023611,POINT (-53.13333 -22.64583)
1,-53.133333,-22.641667,0.023750,POINT (-53.13333 -22.64167)
2,-53.133333,-22.637500,0.023194,POINT (-53.13333 -22.6375)
3,-53.129167,-22.650000,0.023542,POINT (-53.12917 -22.65)
4,-53.129167,-22.645834,0.024028,POINT (-53.12917 -22.64583)
...,...,...,...,...
4219474,-37.354166,-11.595833,0.020347,POINT (-37.35417 -11.59583)
4219475,-37.354166,-11.591667,0.020764,POINT (-37.35417 -11.59167)
4219476,-37.354166,-11.587500,0.021042,POINT (-37.35417 -11.5875)
4219477,-37.354166,-11.583333,0.021806,POINT (-37.35417 -11.58333)


In [6]:
# Perform spatial join
joined_gdf = gpd.sjoin(viirs_cleaned_agg_gdf, ookla_gdf, how='inner', predicate='intersects')

joined_gdf.to_csv('merged_20.csv', index=False)

In [7]:
joined_gdf.head()

,longitude,latitude,light_intensity,geometry,index_right,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices
17,-53.120833,-22.658334,0.024306,POINT (-53.12083 -22.65833),7104907,2103101020113201,42639.0,17910.0,2.0,1,1
95,-53.095833,-22.637500,0.022778,POINT (-53.09583 -22.6375),7104906,2103101020113110,1823.0,5.0,4.0,1,1
99,-53.095833,-22.620834,0.022986,POINT (-53.09583 -22.62083),7104904,2103101020111312,20679.0,23334.0,7.0,1,1
110,-53.091667,-22.637500,0.022292,POINT (-53.09167 -22.6375),7104906,2103101020113110,1823.0,5.0,4.0,1,1
114,-53.091667,-22.620834,0.024097,POINT (-53.09167 -22.62083),7104904,2103101020111312,20679.0,23334.0,7.0,1,1
